# Full mmCIF D-residue universe survey (Colab)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tommaso-R-Marena/ChiralFold/blob/master/demos/Reproduce_mmCIF_D_Residue_Survey.ipynb)

Two cohorts in one run (`--mode both`):

1. **Known errors** — re-verify all **16** legacy-PDB error structures in native mmCIF → same **29** mismatches.
2. **mmCIF-only universe** — discover every live RCSB entry that contains any of the 18 D-amino acid CCD codes and lacks a legacy `.pdb`, then scan with gemmi.

**Why not ~245?** That historical figure counted failed `.pdb` downloads against a broad DPN/DAS/DAL enumeration. Live polymer+ligand CCD search finds **~79** true mmCIF-only entries today — Colab handles the full set in a few minutes.

**Runtime:** ~3–8 minutes (RCSB search + HEAD filter + CIF downloads ≤25 MB each).


In [ ]:
# Cell 1 — Install gemmi + numpy
!pip -q install gemmi numpy pandas


In [ ]:
# Cell 2 — Clone repo for scripts + frozen results
import os, subprocess
if not os.path.isdir('benchmarks'):
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/Tommaso-R-Marena/ChiralFold.git', 'ChiralFold'], check=True)
    os.chdir('ChiralFold')
print('cwd:', os.getcwd())


In [ ]:
# Cell 3 — Full survey: known-errors + live mmCIF-only universe
# Equivalent CLI: python benchmarks/mmcif_d_residue_expansion.py --mode both
!python benchmarks/mmcif_d_residue_expansion.py --mode both


In [ ]:
# Cell 4 — Summarize vs frozen legacy survey
import json
from pathlib import Path
import pandas as pd

legacy = json.loads(Path('results/d_residue_verification_summary.json').read_text(encoding='utf-8'))
mmcif = json.loads(Path('results/mmcif_d_residue_expansion_summary.json').read_text(encoding='utf-8'))
disc = json.loads(Path('results/mmcif_only_universe_ids.json').read_text(encoding='utf-8'))
df = pd.read_csv('results/mmcif_d_residue_expansion.csv')

known = mmcif['known_error_cohort']
uni = mmcif['universe_cohort']
print('Legacy survey errors:', legacy['l_error'], 'in', len(legacy['errors_by_structure']), 'structures')
print('Known-error mmCIF cohort:', known['n_errors'], 'errors in', known['n_structures'], 'structures')
print('  match legacy:', mmcif.get('matches_legacy_survey_errors'))
print('mmCIF-only universe: discovered', disc['n_mmcif_only'], 'of', disc['n_universe_entries'], 'D-AA entries')
print('  scanned errors:', uni['n_errors'], uni['error_pdbs'])
print(df.groupby('cohort')['is_error'].sum() if 'cohort' in df.columns else df['is_error'].sum())
assert known['n_errors'] == 29
assert set(known['error_pdbs']) == set(legacy['errors_by_structure'])
print('OK — known errors recovered; universe survey complete.')
